In [ ]:
# 필수 라이브러리 설치
!pip install transformers torch torchvision pillow

# 내 구글 드라이브 연동 (실행하면 팝업창이 뜨며 권한 허용을 눌러야 합니다)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from torch.cuda.amp import autocast, GradScaler # 🌟 메모리 절약을 위한 마법의 라이브러리

class SatelliteDataset(Dataset):
    def __init__(self, jsonl_file, img_dir, processor):
        self.data = []
        with open(jsonl_file, 'r', encoding='utf-8') as f:
            for line in f:
                self.data.append(json.loads(line))

        self.img_dir = img_dir
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img_path = f"{self.img_dir}/{item['file_name']}"
        image = Image.open(img_path).convert("RGB")
        text = item['text']

        inputs = self.processor(
            text=text,
            images=image,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=77
        )
        return {k: v.squeeze(0) for k, v in inputs.items()}

def train_colab():
    # 1. 한국어 특화 Large 모델 로드
    model_id = "Bingsu/clip-vit-large-patch14-ko"
    processor = CLIPProcessor.from_pretrained(model_id)
    model = CLIPModel.from_pretrained(model_id)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # 2. 구글 드라이브 경로 설정 (본인의 경로에 맞게 확인해주세요)
    # 기본적으로 '내 드라이브/Project/data/raw/tiles/...' 로 설정했습니다.
    jsonl_path = "/content/drive/MyDrive/CV_Project/data/raw/tiles/metadata_solar_test.jsonl"
    img_dir_path = "/content/drive/MyDrive/CV_Project/data/raw/tiles"

    dataset = SatelliteDataset(jsonl_path, img_dir_path, processor)

    # Large 모델 + T4 GPU 조합에서는 batch_size 8이 가장 안정적입니다.
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5) # Large 모델은 학습률을 더 작게 줍니다
    scaler = GradScaler() # FP16 스케일러

    epochs = 5
    model.train()

    print(f"🚀 Colab T4 GPU에서 Large 모델 학습을 시작합니다!")

    for epoch in range(epochs):
        total_loss = 0
        for batch_idx, batch in enumerate(dataloader):
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)

            # 🌟 VRAM 절약 핵심: 연산을 16비트로 가볍게 처리
            with autocast():
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    pixel_values=pixel_values,
                    return_loss=True
                )
                loss = outputs.loss

            # 혼합 정밀도 역전파
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

            if batch_idx % 2 == 0: # 데이터가 적으니 자주 출력해서 확인
                print(f"Epoch [{epoch+1}/{epochs}] Batch [{batch_idx}/{len(dataloader)}] Loss: {loss.item():.4f}")

        print(f"✅ Epoch {epoch+1} 평균 Loss: {total_loss/len(dataloader):.4f}")

    # 3. 내 구글 드라이브에 안전하게 모델 저장
    save_path = "/content/drive/MyDrive/CV_Project/my_large_satellite_clip"
    model.save_pretrained(save_path)
    processor.save_pretrained(save_path)
    print(f"🎉 학습 완료! 구글 드라이브({save_path})에 모델이 저장되었습니다.")

if __name__ == "__main__":
    train_colab()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

...

🚀 Colab T4 GPU에서 Large 모델 학습을 시작합니다!


/tmp/ipykernel_2360/2335951873.py:74: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/5] Batch [0/228] Loss: 2.3255
Epoch [1/5] Batch [2/228] Loss: 2.2827
Epoch [1/5] Batch [4/228] Loss: 3.4670
Epoch [1/5] Batch [6/228] Loss: 2.6778
Epoch [1/5] Batch [8/228] Loss: 1.8360
Epoch [1/5] Batch [10/228] Loss: 1.8919
Epoch [1/5] Batch [12/228] Loss: 2.1058
Epoch [1/5] Batch [14/228] Loss: 1.8277
Epoch [1/5] Batch [16/228] Loss: 1.7126
Epoch [1/5] Batch [18/228] Loss: 1.9165
Epoch [1/5] Batch [20/228] Loss: 2.1664
Epoch [1/5] Batch [22/228] Loss: 2.1862
Epoch [1/5] Batch [24/228] Loss: 2.0321
Epoch [1/5] Batch [26/228] Loss: 1.6850
Epoch [1/5] Batch [28/228] Loss: 1.6533
Epoch [1/5] Batch [30/228] Loss: 1.9012
Epoch [1/5] Batch [32/228] Loss: 1.5238
Epoch [1/5] Batch [34/228] Loss: 1.5607
Epoch [1/5] Batch [36/228] Loss: 1.8888
Epoch [1/5] Batch [38/228] Loss: 1.5397
Epoch [1/5] Batch [40/228] Loss: 1.5514
Epoch [1/5] Batch [42/228] Loss: 1.4622
Epoch [1/5] Batch [44/228] Loss: 1.4027
Epoch [1/5] Batch [46/228] Loss: 2.1798
Epoch [1/5] Batch [48/228] Loss: 1.9156
Epoch

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 학습 완료! 구글 드라이브(/content/drive/MyDrive/Project/my_large_satellite_clip)에 모델이 저장되었습니다.


In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# 1. 방금 학습시킨 '나만의 모델' 불러오기
save_path = "/content/drive/MyDrive/CV_Project/my_large_satellite_clip"
print("가중치 불러오는 중...")
model = CLIPModel.from_pretrained(save_path)
processor = CLIPProcessor.from_pretrained(save_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval() # 평가 모드 전환

# 2. 테스트할 위성 사진 선택 (데이터셋에 있던 101512.jpg로 테스트)
# ⚠️ 경로가 맞는지 확인해 주세요! (로컬로 복사했다면 /content/tiles/...)
test_img_path = "/content/drive/MyDrive/CV_Project/data/raw/tiles/tile_18_223561_101512.jpg"
image = Image.open(test_img_path).convert("RGB")

# 3. 모델에게 물어볼 '후보 객관식(상권 유형)' 텍스트 리스트
candidate_labels = [
    "일반 길거리 상권의 위성 사진입니다.",
    "역세권 생활밀착형 상권의 위성 사진입니다.",
    "대형 랜드마크 주변 상권의 위성 사진입니다.",
    "특별한 상업/공공 시설이 식별되지 않는 일반 도로, 녹지 또는 주택/건물 지붕의 위성 사진"
]

# 4. 이미지와 텍스트를 모델에 넣고 유사도 계산
inputs = processor(
    text=candidate_labels,
    images=image,
    return_tensors="pt",
    padding=True
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

    # 이미지와 각 텍스트 간의 매칭 확률 계산 (Softmax)
    logits_per_image = outputs.logits_per_image
    probs = F.softmax(logits_per_image, dim=1).squeeze().cpu().numpy()

# 5. 결과 출력!
print("\n🔍 [AI 분석 결과]")
for label, prob in zip(candidate_labels, probs):
    print(f"[{prob*100:5.2f}%] {label}")

가중치 불러오는 중...


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]


🔍 [AI 분석 결과]
[77.71%] 일반 길거리 상권의 위성 사진입니다.
[16.99%] 역세권 생활밀착형 상권의 위성 사진입니다.
[ 2.71%] 대형 랜드마크 주변 상권의 위성 사진입니다.
[ 2.59%] 특별한 상업/공공 시설이 식별되지 않는 일반 도로, 녹지 또는 주택/건물 지붕의 위성 사진
